In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings as ws
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
ws.filterwarnings('ignore')



def fim_feature_selection(indep_X,dep_Y,n):
   models = [
        ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=0)),
        ("LinearSVC",  LinearSVC(random_state=0, max_iter=10000)),
        ("DecisionTreeClassifier", DecisionTreeClassifier(criterion='entropy', random_state=0)),
        ("RandomForestClassifier", RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)),
        ("GradientBoostingClassifier", GradientBoostingClassifier(random_state=0)),
           ]
   fimlist = []
   for name, model in models:
        #trains the model using your features indep_X and target dep_Y.
        model.fit(indep_X, dep_Y)

        #Checks whether the model provides built-in feature importance scores (used by tree-based models like Decision Trees and Random Forests).
        
        if hasattr(model, "feature_importances_"):
            importances = pd.Series(model.feature_importances_, index=indep_X.columns)
            
        #Checks whether the model provides coefficients, which are used by linear models such as Linear Regression, Ridge, and Lasso.
        elif hasattr(model, "coef_"):
            
            #gets the learned coefficients from the model.
            coef = model.coef_

            #Converts a 2D coefficient array into a 1D array to match the feature list.
            if coef.ndim > 1:
                coef = coef.ravel()
                #Converts the absolute coefficients into a labeled Series with an importance score for each feature.
                importances = pd.Series(abs(coef), index=indep_X.columns)
        # Handles models without built-in feature importance or coefficients (e.g., SVR).
        else:
            #Computes feature importance by measuring the performance drop after shuffling each feature.
            perm = permutation_importance(model, indep_X, dep_Y, n_repeats=10, random_state=42)
            importances = pd.Series(perm.importances_mean, index=indep_X.columns)

        # #Sorts features by importance and selects the top `n` feature names.
        selected_cols = importances.sort_values(ascending=False).head(n).index.tolist()
        #Creates a new DataFrame with only the top selected features.
        selected_features = indep_X[selected_cols]
        fimlist.append((name, selected_features))
   return fimlist
    
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# cm_prediction - used for classification method, model prediction evaluate confusion matrix and send acciracy, Report and send back 
def cm_prediction(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)

    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred)

    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report

    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return classifier, Accuracy, report, X_test, y_test, cm

    
# logistic method is used for Linear regression model creation and r2 prediction
def logistic(X_train, y_train, X_test, y_test):
    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state=0, max_iter=1000)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def svm_linear(X_train, y_train, X_test, y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='linear', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def svm_NL(X_train, y_train, X_test, y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='rbf', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def Navie(X_train, y_train, X_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def knn(X_train, y_train, X_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def Decision(X_train, y_train, X_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def random_forest(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def gradient_boost(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import GradientBoostingClassifier
    classifier = GradientBoostingClassifier(random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

    
# RFE_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def featureIm_Classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf, accgb): 

    dataframe = pd.DataFrame(
        index=['Logistic', 'LinearSVC', 'DecisionTree', 'RandomForest', 'GradientBoosting'],
        columns=['Logistic', 'SVMl', 'SVMnl', 'KNN', 'Navie', 'Decision', 'Random', 'GradientBoosting']
    )
    print(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf, accgb)
    
    for number, idex in enumerate(dataframe.index):
        dataframe['Logistic'][idex] = acclog[number]
        dataframe['SVMl'][idex] = accsvml[number]
        dataframe['SVMnl'][idex] = accsvmnl[number]
        dataframe['KNN'][idex] = accknn[number]
        dataframe['Navie'][idex] = accnav[number]
        dataframe['Decision'][idex] = accdes[number]
        dataframe['Random'][idex] = accrf[number]
        dataframe['GradientBoosting'][idex] = accgb[number]    
    return dataframe
    

In [32]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using n feature 
featureImList=fim_feature_selection(indep_X,dep_Y,11)      


In [33]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send RFE_regression funtion
# finally the evalution data represent by table view.
acclog=[]
accsSVML=[]
accsSVMnl=[]
accKNN=[]
accNavie=[]
accdes=[]
accrf=[]
accrGB=[]


for name, features_df in featureImList:
    print(name, features_df.shape)
    
    X_train, X_test, y_train, y_test = split_scalar(features_df, dep_Y)

    _, Accuracy, report, _, _, cm = logistic(X_train, y_train, X_test, y_test)
    acclog.append(Accuracy)

    _, Accuracy, report, _, _, cm = svm_linear(X_train, y_train, X_test, y_test)
    accsSVML.append(Accuracy)

    _, Accuracy, report, _, _, cm = svm_NL(X_train, y_train, X_test, y_test)
    accsSVMnl.append(Accuracy)

    _, Accuracy, report, _, _, cm = knn(X_train, y_train, X_test, y_test)
    accKNN.append(Accuracy)

    _, Accuracy, report, _, _, cm = Navie(X_train, y_train, X_test, y_test)
    accNavie.append(Accuracy)

    _, Accuracy, report, _, _, cm = Decision(X_train, y_train, X_test, y_test)
    accdes.append(Accuracy)

    _, Accuracy, report, _, _, cm = random_forest(X_train, y_train, X_test, y_test)
    accrf.append(Accuracy)

    _, Accuracy, report, _, _, cm = gradient_boost(X_train, y_train, X_test, y_test)
    accrGB.append(Accuracy)

result = featureIm_Classification(acclog, accsSVML, accsSVMnl, accKNN, accNavie, accdes, accrf, accrGB)


LogisticRegression (399, 11)
LinearSVC (399, 11)
DecisionTreeClassifier (399, 11)
RandomForestClassifier (399, 11)
GradientBoostingClassifier (399, 11)
[0.99, 0.98, 0.98, 1.0, 0.98] [0.98, 0.97, 0.97, 0.97, 0.99] [0.99, 0.98, 0.96, 1.0, 0.99] [0.98, 0.98, 0.97, 1.0, 0.99] [0.98, 0.98, 0.94, 0.96, 0.98] [0.96, 0.96, 0.95, 0.96, 0.95] [1.0, 1.0, 1.0, 0.98, 0.99] [1.0, 1.0, 1.0, 0.98, 1.0]


In [25]:
result
# 8

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random,GradientBoosting
Logistic,0.98,0.98,0.98,0.96,0.98,0.97,1.0,1.0
LinearSVC,0.96,0.96,0.95,0.96,0.95,0.95,0.97,1.0
DecisionTree,1.0,1.0,0.98,0.99,0.93,0.96,0.96,0.99
RandomForest,0.98,0.98,0.99,0.98,0.9,0.97,0.98,0.98
GradientBoosting,0.97,0.97,0.98,0.98,0.94,0.98,0.96,1.0


In [28]:
result
# 9

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random,GradientBoosting
Logistic,0.99,0.98,0.98,0.95,0.98,0.95,1.0,1.0
LinearSVC,0.98,0.98,0.98,0.97,0.98,0.98,1.0,1.0
DecisionTree,0.98,1.0,0.97,0.97,0.93,0.96,0.99,0.99
RandomForest,0.99,0.97,1.0,1.0,0.93,0.96,1.0,0.99
GradientBoosting,0.98,0.98,0.99,0.98,0.95,0.98,0.99,1.0


In [31]:
result
#10

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random,GradientBoosting
Logistic,0.98,0.97,0.98,0.97,0.98,0.96,0.99,1.0
LinearSVC,0.98,0.97,0.98,0.97,0.98,0.95,0.99,1.0
DecisionTree,0.97,0.97,0.96,0.98,0.93,0.96,1.0,0.99
RandomForest,1.0,0.97,1.0,1.0,0.96,0.96,0.98,1.0
GradientBoosting,0.98,0.99,0.99,0.99,0.98,0.95,0.99,1.0


In [34]:
result
# 11

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random,GradientBoosting
Logistic,0.99,0.98,0.99,0.98,0.98,0.96,1.0,1.0
LinearSVC,0.98,0.97,0.98,0.98,0.98,0.96,1.0,1.0
DecisionTree,0.98,0.97,0.96,0.97,0.94,0.95,1.0,1.0
RandomForest,1.0,0.97,1.0,1.0,0.96,0.96,0.98,0.98
GradientBoosting,0.98,0.99,0.99,0.99,0.98,0.95,0.99,1.0
